# SC-9-DAO-Governance - Gouvernance DAO

[<< DeFi Primitives](SC-8-DeFi-Primitives.ipynb) | [Account Abstraction >>](SC-10-Account-Abstraction.ipynb)

---

## Objectifs d'apprentissage

1. Implementer un systeme de **vote** pondere
2. Creer et executer des **propositions**
3. Utiliser un **timelock** pour la securite

### Prerequis

- [SC-1](../01-Solidity-Foundation/SC-3-Solidity-Basics.ipynb) a [SC-6](SC-8-DeFi-Primitives.ipynb) completes
- Concepts de gouvernance on-chain et votes
- Tokens ERC-20 et delégation (SC-5)

### Duree estimee : 45 minutes

---

## 0. Connexion a la blockchain locale

Tous les contrats de ce notebook sont **compiles et deployes reellement** sur anvil.
Lancez `anvil` dans un terminal avant d'executer les cellules.


In [1]:
# Connection a anvil (blockchain locale Foundry)
# Prerequis: anvil en cours d'execution dans un terminal
from web3 import Web3
import solcx

SOLC_VERSION = "0.8.28"
ANVIL_URL = "http://127.0.0.1:8545"

# Connexion
w3 = Web3(Web3.HTTPProvider(ANVIL_URL))
assert w3.is_connected(), f"Impossible de se connecter a {ANVIL_URL}. Lancez 'anvil' dans un terminal."

# Reset anvil pour un etat propre (les checkpoints de vote dependent du block.number)
w3.provider.make_request("anvil_reset", [])

# Installer solc si necessaire
installed = [str(v) for v in solcx.get_installed_solc_versions()]
if SOLC_VERSION not in installed:
    solcx.install_solc(SOLC_VERSION)
solcx.set_solc_version(SOLC_VERSION)

deployer = w3.eth.accounts[0]
print(f"Connecte a anvil (chain {w3.eth.chain_id}), bloc {w3.eth.block_number}, deployer: {deployer[:10]}...")


def compile_and_deploy(w3, source_code, deployer, *constructor_args):
    """Compiler et deployer un contrat Solidity."""
    compiled = solcx.compile_source(
        source_code, output_values=["abi", "bin"], solc_version=SOLC_VERSION
    )
    contract_id, contract_interface = None, None
    for cid, ci in compiled.items():
        if ci["bin"]:
            contract_id, contract_interface = cid, ci
    if contract_interface is None:
        contract_id, contract_interface = compiled.popitem()
    Contract = w3.eth.contract(
        abi=contract_interface["abi"], bytecode=contract_interface["bin"]
    )
    tx_hash = Contract.constructor(*constructor_args).transact({"from": deployer})
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    instance = w3.eth.contract(
        address=receipt.contractAddress, abi=contract_interface["abi"]
    )
    print(f"Deploye: {contract_id.split(':')[-1]} a {receipt.contractAddress}")
    return instance, receipt

Connecte a anvil (chain 31337), bloc 0, deployer: 0xf39Fd6e5...


---

## 1. Governance Token

Le token de gouvernance donne des droits de vote.

In [2]:
# Governance Token avec checkpoint de votes
GOVERNANCE_TOKEN = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract GovernanceToken {
    string public constant name = "Governance Token";
    string public constant symbol = "GOV";
    uint8 public constant decimals = 18;

    uint256 private _totalSupply;
    mapping(address => uint256) private _balances;
    mapping(address => mapping(address => uint256)) private _allowances;

    struct Checkpoint {
        uint32 fromBlock;
        uint224 votes;
    }

    mapping(address => Checkpoint[]) private _checkpoints;
    mapping(address => address) private _delegates;

    event Transfer(address indexed from, address indexed to, uint256 value);
    event DelegateChanged(address indexed delegator, address indexed fromDelegate, address indexed toDelegate);

    constructor(uint256 initialSupply) {
        _mint(msg.sender, initialSupply);
    }

    function delegate(address delegatee) public {
        address currentDelegate = delegates(msg.sender);
        uint256 balance = _balances[msg.sender];
        _delegates[msg.sender] = delegatee;
        _moveVotingPower(currentDelegate, delegatee, balance);
        emit DelegateChanged(msg.sender, currentDelegate, delegatee);
    }

    function delegates(address account) public view returns (address) {
        address d = _delegates[account];
        return d == address(0) ? account : d;
    }

    function getVotes(address account) public view returns (uint256) {
        uint256 pos = _checkpoints[account].length;
        return pos > 0 ? _checkpoints[account][pos - 1].votes : 0;
    }

    function getPastVotes(address account, uint256 blockNumber)
        public view returns (uint256)
    {
        require(blockNumber < block.number, "Block not yet mined");
        Checkpoint[] storage ckpts = _checkpoints[account];
        uint256 low = 0;
        uint256 high = ckpts.length;
        while (low < high) {
            uint256 mid = (low + high) / 2;
            if (ckpts[mid].fromBlock > blockNumber) {
                high = mid;
            } else {
                low = mid + 1;
            }
        }
        return high == 0 ? 0 : ckpts[high - 1].votes;
    }

    function totalSupply() public view returns (uint256) { return _totalSupply; }
    function balanceOf(address account) public view returns (uint256) { return _balances[account]; }

    function transfer(address to, uint256 amount) public returns (bool) {
        require(_balances[msg.sender] >= amount, "Insufficient balance");
        _balances[msg.sender] -= amount;
        _balances[to] += amount;
        _moveVotingPower(delegates(msg.sender), delegates(to), amount);
        emit Transfer(msg.sender, to, amount);
        return true;
    }

    function _mint(address to, uint256 amount) internal {
        _totalSupply += amount;
        _balances[to] += amount;
        _moveVotingPower(address(0), delegates(to), amount);
        emit Transfer(address(0), to, amount);
    }

    function _moveVotingPower(address src, address dst, uint256 amount) internal {
        if (src != address(0)) {
            uint256 pos = _checkpoints[src].length;
            uint224 old = pos > 0 ? _checkpoints[src][pos - 1].votes : 0;
            _writeCheckpoint(_checkpoints[src], old - uint224(amount));
        }
        if (dst != address(0)) {
            uint256 pos = _checkpoints[dst].length;
            uint224 old = pos > 0 ? _checkpoints[dst][pos - 1].votes : 0;
            _writeCheckpoint(_checkpoints[dst], old + uint224(amount));
        }
    }

    function _writeCheckpoint(Checkpoint[] storage ckpts, uint224 newVotes) internal {
        uint256 pos = ckpts.length;
        if (pos > 0 && ckpts[pos - 1].fromBlock == block.number) {
            ckpts[pos - 1].votes = newVotes;
        } else {
            ckpts.push(Checkpoint({fromBlock: uint32(block.number), votes: newVotes}));
        }
    }
}
'''

# Deployer le GovernanceToken avec 10M tokens
initial_supply = 10_000_000 * 10**18
gov_token, _ = compile_and_deploy(w3, GOVERNANCE_TOKEN, deployer, initial_supply)

# Deleguer le voting power a soi-meme
tx = gov_token.functions.delegate(deployer).transact({"from": deployer})
w3.eth.wait_for_transaction_receipt(tx)

# Transferer des tokens et deleguer pour un second votant
voter2 = w3.eth.accounts[1]
tx = gov_token.functions.transfer(voter2, 2_000_000 * 10**18).transact({"from": deployer})
w3.eth.wait_for_transaction_receipt(tx)
tx = gov_token.functions.delegate(voter2).transact({"from": voter2})
w3.eth.wait_for_transaction_receipt(tx)

# Miner des blocs pour ancrer les checkpoints (getPastVotes regarde block.number - 1)
for _ in range(3):
    w3.provider.make_request("evm_mine", [])

print(f"Governance Token:")
print(f"  Total supply   : {gov_token.functions.totalSupply().call() / 10**18:,.0f} GOV")
print(f"  Votes deployer : {gov_token.functions.getVotes(deployer).call() / 10**18:,.0f} GOV")
print(f"  Votes voter2   : {gov_token.functions.getVotes(voter2).call() / 10**18:,.0f} GOV")

Deploye: GovernanceToken a 0x5FbDB2315678afecb367f032d93F642f64180aa3
Governance Token:
  Total supply   : 10,000,000 GOV
  Votes deployer : 8,000,000 GOV
  Votes voter2   : 2,000,000 GOV


---

## 2. Governor Contract

Le governor gere le cycle de vie des propositions.

In [3]:
# Governor simplifie
GOVERNOR_CONTRACT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

interface IGovernanceToken {
    function getVotes(address account) external view returns (uint256);
    function getPastVotes(address account, uint256 blockNumber) external view returns (uint256);
}

contract SimpleGovernor {
    IGovernanceToken public immutable token;

    uint256 public constant VOTING_DELAY = 1;
    uint256 public constant VOTING_PERIOD = 50400;
    uint256 public constant PROPOSAL_THRESHOLD = 100 * 1e18;
    uint256 public constant QUORUM = 4_000_000 * 1e18;

    enum ProposalState { Pending, Active, Canceled, Defeated, Succeeded, Queued, Expired, Executed }

    struct Proposal {
        uint256 id;
        address proposer;
        address[] targets;
        uint256[] values;
        bytes[] calldatas;
        string description;
        uint256 voteStart;
        uint256 voteEnd;
        uint256 forVotes;
        uint256 againstVotes;
        uint256 abstainVotes;
        bool executed;
    }

    mapping(uint256 => Proposal) public proposals;
    mapping(uint256 => mapping(address => bool)) public hasVoted;
    uint256 private _proposalCount;

    event ProposalCreated(uint256 indexed id, address indexed proposer, address[] targets, uint256[] values, bytes[] calldatas, string description);
    event VoteCast(address indexed voter, uint256 indexed proposalId, uint8 support, uint256 weight);

    constructor(address token_) { token = IGovernanceToken(token_); }

    function propose(address[] memory targets, uint256[] memory values, bytes[] memory calldatas, string memory description) public returns (uint256) {
        require(token.getPastVotes(msg.sender, block.number - 1) >= PROPOSAL_THRESHOLD, "Below proposal threshold");
        uint256 proposalId = ++_proposalCount;
        proposals[proposalId] = Proposal({
            id: proposalId, proposer: msg.sender, targets: targets, values: values,
            calldatas: calldatas, description: description,
            voteStart: block.number + VOTING_DELAY, voteEnd: block.number + VOTING_DELAY + VOTING_PERIOD,
            forVotes: 0, againstVotes: 0, abstainVotes: 0, executed: false
        });
        emit ProposalCreated(proposalId, msg.sender, targets, values, calldatas, description);
        return proposalId;
    }

    function castVote(uint256 proposalId, uint8 support) public {
        require(state(proposalId) == ProposalState.Active, "Voting not active");
        require(!hasVoted[proposalId][msg.sender], "Already voted");
        Proposal storage proposal = proposals[proposalId];
        uint256 weight = token.getPastVotes(msg.sender, proposal.voteStart);
        hasVoted[proposalId][msg.sender] = true;
        if (support == 0) proposal.againstVotes += weight;
        else if (support == 1) proposal.forVotes += weight;
        else proposal.abstainVotes += weight;
        emit VoteCast(msg.sender, proposalId, support, weight);
    }

    function state(uint256 proposalId) public view returns (ProposalState) {
        Proposal storage p = proposals[proposalId];
        if (p.executed) return ProposalState.Executed;
        if (block.number < p.voteStart) return ProposalState.Pending;
        if (block.number < p.voteEnd) return ProposalState.Active;
        uint256 total = p.forVotes + p.againstVotes + p.abstainVotes;
        if (total < QUORUM || p.againstVotes >= p.forVotes) return ProposalState.Defeated;
        return ProposalState.Succeeded;
    }
}
'''

# Deployer le Governor
governor, _ = compile_and_deploy(w3, GOVERNOR_CONTRACT, deployer, gov_token.address)

# Miner des blocs pour que les checkpoints de delegation soient bien dans le passe
for _ in range(3):
    w3.provider.make_request("evm_mine", [])

# Creer une proposition
tx = governor.functions.propose(
    [deployer], [0], [b""],
    "Proposition #1: Augmenter le budget R&D de 10%"
).transact({"from": deployer})
receipt = w3.eth.wait_for_transaction_receipt(tx)
proposal_events = governor.events.ProposalCreated().process_receipt(receipt)
proposal_id = proposal_events[0]["args"]["id"]

state_names = ["Pending", "Active", "Canceled", "Defeated", "Succeeded", "Queued", "Expired", "Executed"]
print(f"\nGovernor:")
print(f"  Proposal ID : {proposal_id}")
print(f"  Etat        : {state_names[governor.functions.state(proposal_id).call()]}")

# Avancer de 2 blocs : 1 pour VOTING_DELAY, 1 pour que voteStart soit dans le passe
# (castVote appelle getPastVotes(voter, voteStart) qui exige voteStart < block.number)
w3.provider.make_request("evm_mine", [])
w3.provider.make_request("evm_mine", [])
print(f"  Etat        : {state_names[governor.functions.state(proposal_id).call()]}")

# Voter
tx = governor.functions.castVote(proposal_id, 1).transact({"from": deployer})
w3.eth.wait_for_transaction_receipt(tx)
tx = governor.functions.castVote(proposal_id, 1).transact({"from": voter2})
w3.eth.wait_for_transaction_receipt(tx)

# Resultats
proposal_data = governor.functions.proposals(proposal_id).call()
print(f"\nVotes:")
print(f"  Pour    : {proposal_data[5] / 10**18:,.0f} GOV")
print(f"  Contre  : {proposal_data[6] / 10**18:,.0f} GOV")
print(f"  Abstain : {proposal_data[7] / 10**18:,.0f} GOV")

Deploye: SimpleGovernor a 0xCf7Ed3AccA5a467e9e704C703E8D87F634fB0Fc9

Governor:
  Proposal ID : 1
  Etat        : Pending
  Etat        : Active



Votes:
  Pour    : 10,000,000 GOV
  Contre  : 0 GOV
  Abstain : 0 GOV


---

## 3. Timelock Controller

Le timelock ajoute un delai avant l'execution pour la securite.

In [4]:
# Timelock simplifie
TIMELOCK_CONTRACT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract SimpleTimelock {
    uint256 public constant MIN_DELAY = 1 days;
    uint256 public constant MAX_DELAY = 30 days;
    uint256 public constant GRACE_PERIOD = 14 days;

    address public admin;
    uint256 public delay;

    mapping(bytes32 => bool) public queuedTransactions;

    event QueueTransaction(
        bytes32 indexed txHash,
        address indexed target,
        uint256 value,
        bytes data,
        uint256 eta
    );
    event ExecuteTransaction(
        bytes32 indexed txHash,
        address indexed target,
        uint256 value,
        bytes data
    );
    event CancelTransaction(bytes32 indexed txHash);

    constructor(uint256 delay_) {
        require(delay_ >= MIN_DELAY, "Delay too short");
        require(delay_ <= MAX_DELAY, "Delay too long");
        admin = msg.sender;
        delay = delay_;
    }

    // Hash unique pour chaque transaction
    function getTxHash(
        address target,
        uint256 value,
        bytes calldata data,
        uint256 eta
    ) public pure returns (bytes32) {
        return keccak256(abi.encode(target, value, data, eta));
    }

    // Mettre en file d'attente
    function queue(
        address target,
        uint256 value,
        bytes calldata data,
        uint256 eta
    ) public returns (bytes32) {
        require(msg.sender == admin, "Not admin");
        require(eta >= block.timestamp + delay, "ETA too soon");

        bytes32 txHash = getTxHash(target, value, data, eta);
        require(!queuedTransactions[txHash], "Already queued");

        queuedTransactions[txHash] = true;
        emit QueueTransaction(txHash, target, value, data, eta);

        return txHash;
    }

    // Executer apres le delai
    function execute(
        address target,
        uint256 value,
        bytes calldata data,
        uint256 eta
    ) public payable {
        require(msg.sender == admin, "Not admin");

        bytes32 txHash = getTxHash(target, value, data, eta);
        require(queuedTransactions[txHash], "Not queued");
        require(block.timestamp >= eta, "Too early");
        require(block.timestamp <= eta + GRACE_PERIOD, "Expired");

        queuedTransactions[txHash] = false;

        (bool success, ) = target.call{value: value}(data);
        require(success, "Execution failed");

        emit ExecuteTransaction(txHash, target, value, data);
    }

    // Annuler
    function cancel(bytes32 txHash) public {
        require(msg.sender == admin, "Not admin");
        require(queuedTransactions[txHash], "Not queued");

        queuedTransactions[txHash] = false;
        emit CancelTransaction(txHash);
    }
}
'''

# Deployer le Timelock avec un delai de 1 jour (86400 secondes)
timelock, _ = compile_and_deploy(w3, TIMELOCK_CONTRACT, deployer, 86400)

# --- Demonstration: mettre une transaction en file d'attente ---
print(f"\nTimelock Controller:")
print(f"  Admin     : {timelock.functions.admin().call()[:10]}...")
print(f"  Delay     : {timelock.functions.delay().call()} secondes ({timelock.functions.delay().call() // 3600}h)")
print(f"  Min delay : {timelock.functions.MIN_DELAY().call()} secondes")
print(f"  Max delay : {timelock.functions.MAX_DELAY().call()} secondes")

# Calculer l'ETA (timestamp actuel + delay + marge)
current_block = w3.eth.get_block("latest")
current_timestamp = current_block["timestamp"]
eta = current_timestamp + 86400 + 60  # delay + 1 minute de marge

# Mettre en queue une transaction fictive (envoyer 0 ETH a deployer)
tx = timelock.functions.queue(
    deployer,  # target
    0,         # value
    b"",       # data (vide)
    eta        # eta
).transact({"from": deployer})
receipt = w3.eth.wait_for_transaction_receipt(tx)

# Verifier le hash
tx_hash = timelock.functions.getTxHash(deployer, 0, b"", eta).call()
is_queued = timelock.functions.queuedTransactions(tx_hash).call()
print(f"\n  Transaction en queue: {is_queued}")
print(f"  TX hash   : {tx_hash.hex()[:20]}...")
print(f"  ETA       : timestamp {eta} (dans ~24h)")

# Annuler la transaction
tx = timelock.functions.cancel(tx_hash).transact({"from": deployer})
w3.eth.wait_for_transaction_receipt(tx)
is_queued_after = timelock.functions.queuedTransactions(tx_hash).call()
print(f"  Apres cancel: queued={is_queued_after}")

Deploye: SimpleTimelock a 0x0165878A594ca255338adfa4d48449f69242Eb8F

Timelock Controller:
  Admin     : 0xf39Fd6e5...
  Delay     : 86400 secondes (24h)
  Min delay : 86400 secondes
  Max delay : 2592000 secondes

  Transaction en queue: True
  TX hash   : 09ab826363db047fdd6f...
  ETA       : timestamp 1776692872 (dans ~24h)


  Apres cancel: queued=False


---

## 4. Exercices

### Exercice 1 : Calcul du quorum

Ecrivez une fonction pour verifier si le quorum est atteint.

In [5]:
# Exercice 1 : Calcul du quorum
def check_quorum(for_votes, against_votes, abstain_votes, total_supply, quorum_percent=4):
    """
    Verifie si le quorum est atteint
    
    Quorum = (for + against + abstain) >= quorum_percent * total_supply / 100
    
    Args:
        for_votes: Nombre de votes pour
        against_votes: Nombre de votes contre
        abstain_votes: Nombre d abstentions
        total_supply: Supply totale de tokens
        quorum_percent: Pourcentage requis pour le quorum
    
    Returns:
        dict avec:
        - quorum_reached (bool)
        - total_votes
        - quorum_threshold
        - participation_percent
        - for_percent
        - against_percent
    """
    # TODO: Calculer le total des votes
    # TODO: Calculer le seuil du quorum (total_supply * quorum_percent / 100)
    # TODO: Determiner si le quorum est atteint
    # TODO: Calculer le pourcentage de participation
    # TODO: Calculer les pourcentages pour/contre
    # TODO: Retourner le dictionnaire avec tous les resultats
    pass

# Test (decommenter apres implementation)
# total_supply = 100_000_000 * 1e18  # 100M tokens
# result = check_quorum(
#     for_votes=3_500_000 * 1e18,
#     against_votes=1_000_000 * 1e18,
#     abstain_votes=500_000 * 1e18,
#     total_supply=total_supply
# )
# 
# print(f"Quorum atteint: {result["quorum_reached"]}")
# print(f"Participation: {result["participation_percent"]:.2f}%")
# print(f"Pour: {result["for_percent"]:.1f}%")
# print(f"Contre: {result["against_percent"]:.1f}%")

### Exercice 2 : Simulation de vote

Simulez un processus de gouvernance complet.

In [6]:
# Exercice 2 : Simulation de vote
class ProposalSimulation:
    """Simule un processus de gouvernance complet."""
    
    def __init__(self, quorum=4_000_000, threshold=0.5):
        """
        Args:
            quorum: Nombre minimum de votes requis
            threshold: Seuil de majorite (0.5 = 50%)
        """
        self.quorum = quorum
        self.threshold = threshold
        # TODO: Initialiser les compteurs de votes (for, against, abstain)
        # TODO: Initialiser un dictionnaire pour tracker les votants

    def vote(self, voter, weight, support):
        """
        Enregistre un vote.
        
        Args:
            voter: Identifiant du votant
            weight: Poids du vote (nombre de tokens)
            support: 0 = contre, 1 = pour, 2 = abstention
        """
        # TODO: Verifier que le votant n a pas deja vote
        # TODO: Enregistrer le vote dans le dictionnaire
        # TODO: Ajouter le poids au compteur correspondant (for/against/abstain)
        pass

    def get_result(self):
        """
        Determine le resultat du vote.
        
        Returns:
            str: "SUCCEEDED", "DEFEATED - Quorum non atteint", 
                 ou "DEFEATED - Majorite contre"
        """
        # TODO: Calculer le total des votes
        # TODO: Verifier si le quorum est atteint
        # TODO: Verifier si la majorite est pour
        # TODO: Retourner le resultat
        pass

# Test (decommenter apres implementation)
# sim = ProposalSimulation()
# sim.vote("Alice", 1_000_000, 1)    # Pour
# sim.vote("Bob", 2_000_000, 1)      # Pour
# sim.vote("Charlie", 1_500_000, 0)  # Contre
# sim.vote("Dave", 500_000, 2)       # Abstain
# sim.vote("Eve", 800_000, 1)        # Pour
# 
# print(f"Pour: {sim.for_votes:,}")
# print(f"Contre: {sim.against_votes:,}")
# print(f"Abstention: {sim.abstain_votes:,}")
# print(f"Resultat: {sim.get_result()}")

---

## 5. Resume

| Composant | Role |
|-----------|------|
| GovernanceToken | Token avec voting power |
| Governor | Gestion des propositions et votes |
| Timelock | Delai de securite avant execution |

### Parametres cles
- **Voting Delay** : Blocs avant debut du vote
- **Voting Period** : Duree du vote
- **Quorum** : Participation minimum requise
- **Proposal Threshold** : Tokens min pour proposer

---

**Notebook suivant** : [SC-10-Account-Abstraction](SC-10-Account-Abstraction.ipynb)

---

[<< DeFi Primitives](SC-8-DeFi-Primitives.ipynb) | [Account Abstraction >>](SC-10-Account-Abstraction.ipynb)